In [1]:
import os, sys
sys.path.append("E:\\Dai hoc\\2526I\\dacn\\flow-matching\\pep2prob")
# os.environ["PYTHONPATH"] = "E:\\Dai hoc\\2526I\\dacn\\flow-matching\\pep2prob"

In [2]:
import torch
from torch.nn.functional import mse_loss
import esm
from flow_matching.path import CondOTProbPath
# from flow_matching.loss import 

In [3]:
from my_model.mlp.mlp import FlowMLP
from p2p_dataset import Pep2ProbDataset


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
device.type
torch.set_default_device('cuda')

In [6]:
peptide_embedding_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
peptide_embedding_model.eval()
batch_converter = alphabet.get_batch_converter()
pep2prob_ds = Pep2ProbDataset(data_dir="data")

Download complete.
Dataset and data split downloaded successfully.


In [ ]:
def extract_ds_row(ds_row: tuple):
    (precursor, probs, prob_masks, info) = ds_row
    return {
        "peptide": precursor[0],
        "charge": precursor[1],
        "probs": probs,     
        "prob_masks": prob_masks,  
        "info": info
    }

def get_peptide_embs(peptides: list[str], device="cuda"):
    batch_labels, batch_strs, batch_tokens = batch_converter(
        [(f"p_{i}", pep) for i, pep in enumerate(peptides)]
    )
    
    batch_tokens = batch_tokens.to(device)
    
    with torch.no_grad():
        results = peptide_embedding_model(
            batch_tokens, 
            repr_layers=[12], 
            return_contacts=False
        )
    
    token_reps = results["representations"][12]
    
    # Efficiently calculate mean embeddings excluding special tokens
    peptide_embeddings = []
    for i, label in enumerate(batch_labels):
        seq_len = len(label)
        emb = token_reps[i, 1 : seq_len + 1].mean(dim=0)
        peptide_embeddings.append(emb)
        
    return torch.stack(peptide_embeddings)

def collate_fn(batch):
    """
    Aggregates data into CUDA-ready batches.
    """
    device = "cuda"
    
    # 1. Unpack sequences
    peptides = [item["peptide"] for item in batch]
    
    # 2. Process NumPy arrays to Tensors on CUDA
    # torch.as_tensor is safer for varying types, .to(device) moves it
    probs = torch.stack([torch.as_tensor(item["probs"]) for item in batch]).to(device)
    prob_masks = torch.stack([torch.as_tensor(item["prob_masks"]) for item in batch]).to(device)
    charges = torch.tensor([item["charge"] for item in batch], dtype=torch.float32, device=device)
    
    # 3. Generate embeddings on GPU
    peptide_embs = get_peptide_embs(peptides, device=device)
    
    # 4. Meta-info remains as a list (usually strings/dicts can't be tensors)
    infos = [item["info"] for item in batch]
    
    return {
        "peptide_embs": peptide_embs,
        "charges": charges,
        "probs": probs,
        "prob_masks": prob_masks,
        "infos": infos
    }

In [8]:
collate_fn([pep2prob_ds.train[0]])

TypeError: tuple indices must be integers or slices, not str